In [ ]:
# 初始化 setting 类, 用pydantic导入各种key
from pydantic import Field
from pydantic_settings import BaseSettings
from dotenv import load_dotenv

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai_like import OpenAILike

from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.node_parser import SemanticSplitterNodeParser


from llama_index.core import Document


load_dotenv()  # 从 .env 文件加载环境变量到系统环境


class Settings(BaseSettings):
    agnes_api_model: str = Field(..., env="AGNES_API_MODEL")
    agnes_api_key: str = Field(..., env="AGNES_API_KEY")
    agnes_api_url: str = Field(..., env="AGNES_API_URL")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"
        case_sensitive = False
        env_prefix = ""
        extra = "ignore"

/Users/pengzishang/miniconda3/envs/LlamaIndex/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/vv/w3jwg8m93_n11f0wcfssmqzr0000gn/T/ipykernel_18060/809088488.py:23: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'env'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  agnes_api_model: str = Field(..., env='AGNES_API_MODEL')
/var/folders/vv/w3jwg8m93_n11f0wcfssmqzr0000gn/T/ipykernel_18060/809088488.py:24: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'env'). Deprecated in Pydantic V2.0 to be removed in V3.0. Se

In [ ]:
!pip install "paddlepaddle<=2.6.2"
# 安装PaddleOCR whl包
!pip install "paddleocr<3.0"
!pip install pydantic-settings
!pip install python-dotenv
!pip install llama-index
!pip install llama-index-core
!pip install llama-index-llms-openai-like
!pip install llama-index-embeddings-huggingface

In [ ]:
# 验证安装是否成功
import paddleocr

print(f"PaddleOCR版本: {paddleocr.__version__}")

# 若使用本地推理引擎 paddle_static，可继续验证 PaddlePaddle 与 GPU 是否可用
import paddle

print(f"Paddle版本: {paddle.__version__}")
print(f"GPU可用: {paddle.is_compiled_with_cuda()}")
print(f"GPU数量: {paddle.device.cuda.device_count()}")

# 若使用 transformers 推理引擎，可继续验证 transformers 依赖是否可用
import transformers

print(f"Transformers版本: {transformers.__version__}")

PaddleOCR版本: 2.10.0
Paddle版本: 2.6.2
GPU可用: False
GPU数量: 0
Transformers版本: 5.12.1


In [ ]:
from paddleocr import PaddleOCR
import cv2

ocr = PaddleOCR(
    use_doc_orientation_classify=False,  # 通过 use_doc_orientation_classify 参数指定不使用文档方向分类模型
    use_doc_unwarping=False,  # 通过 use_doc_unwarping 参数指定不使用文本图像矫正模型
    use_textline_orientation=False,  # 通过 use_textline_orientation 参数指定不使用文本行方向分类模型
)

result = ocr.ocr("./images/general_ocr_003.png")
# 读取原图
# 处理识别结果
all_text = ""
for img_idx, img_result in enumerate(result):
    print(f"\n=== 图片 {img_idx + 1} 的识别结果 ===")

    for word_info in img_result:
        text = word_info[1][0]  # 文字内容
        confidence = word_info[1][1]  # 置信度
        print(f"{text} (置信度: {confidence:.2f})")
        all_text += text + "\n"

# 保存识别结果到文件
with open("output/ocr_result.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print(f"\n✅ 结果已保存到 output/ocr_result.txt")

In [43]:
from llama_index.core.readers.base import BaseReader
from llama_index.core.schema import Document
import os
from typing import List, Union
from paddleocr import PaddleOCR
from PIL import Image
import time


class ImageOCRReader(BaseReader):
    """使用 PP-OCR v5 从图像中提取文本并返回 Document"""

    def __init__(self, lang="ch", **kwargs):
        """
        Args:
            lang: OCR 语言 ('ch', 'en', 'fr', etc.)
            **kwargs: 其他传递给 PaddleOCR 的参数
        """
        self.lang = lang
        self.ocr = PaddleOCR(
            use_doc_orientation_classify=False,  # 通过 use_doc_orientation_classify 参数指定不使用文档方向分类模型
            use_doc_unwarping=False,  # 通过 use_doc_unwarping 参数指定不使用文本图像矫正模型
            use_textline_orientation=False,  # 通过 use_textline_orientation 参数指定不使用文本行方向分类模型
        )

    def load_data(self, file: Union[str, List[str]], **kwargs) -> List[Document]:
        """
        从单个或多个图像文件中提取文本，返回 Document 列表
        Args:
            file: 图像路径字符串 或 路径列表
            **kwargs: 其他传递给 PaddleOCR 的参数
        Returns:
            List[Document]
        """
        # 实现 OCR 提取逻辑
        # 将每张图的识别结果拼接成文本
        # 构造 Document 对象，附带元数据（如 image_path, ocr_confidence_avg）

        if isinstance(file, str):
            file = [file]

        documents = []
        for f in file:
            if not os.path.exists(f):
                raise FileNotFoundError(f"文件不存在: {f}")
            if not f.endswith((".jpg", ".jpeg", ".png", ".bmp")):
                raise ValueError(f"文件格式错误: {f}")

            img = Image.open(f)
            width, height = img.size
            # 读取图像文件
            # with open(f, 'rb') as img:
            #     image_data = img.read()
            # 调用 PaddleOCR 进行识别
            results = self.ocr.ocr(f)
            text = ""
            for line in results:
                for item in line:  # 遍历每个文字区域
                    text += item[1][0] + "\n"  # 识别的文字

                    # 构造 Document 对象
            document = Document(
                text=text,
                metadata={
                    "image_path": f,
                    # 基础信息
                    "file_name": os.path.basename(f),
                    "file_size": os.path.getsize(f),
                    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    # OCR 信息
                    "ocr_lang": "ch",
                    "ocr_word_count": len(text),
                    "ocr_model_version": "PP-OCRv4",
                    # 图片信息
                    "image_width": width,
                    "image_height": height,
                    "image_format": "png",
                },
            )
            documents.append(document)
            return documents

In [47]:
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


reader = ImageOCRReader(lang="ch")
documents = reader.load_data("./images/general_ocr_002.png")

splitter = SentenceSplitter(
    chunk_overlap=30,
    chunk_size=200,
)


splitter = TokenTextSplitter(chunk_size=200, chunk_overlap=10, separator="\n")

splitter = SemanticSplitterNodeParser(
    embed_model=HuggingFaceEmbedding(
        model_name="BAAI/bge-small-zh-v1.5"  # 中文效果好的免费模型
    ),
    breakpoint_percentile_threshold=95,  # 相似度差异超过95%分位才切断
)

nodes = splitter.get_nodes_from_documents(documents)

for node in nodes:
    print(node.text)
    print("----------------")

[2026/06/19 14:31:22] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='/Users/pengzishang/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='/Users/pengzishang/.paddleocr/whl/rec/ch/ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', r

RemoteProtocolError: Server disconnected without sending a response.